In [0]:
df_bronze = spark.read.table("workspace.bronze.bronze_people")

display(df_bronze)

#  Drop index column 

In [0]:
df_silver = df_bronze.drop("index")
display(df_silver)

Cast columns to correct types

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, DateType

# 1. Define your desired schema mapping
# This acts as your "Source of Truth" for the Silver layer
target_schema = {
    "user_id": StringType(),
    "first_name": StringType(),
    "last_name": StringType(),
    "sex": StringType(),
    "email": StringType(),
    "phone": StringType(),
    "date_of_birth": DateType(),
    "job_title": StringType()
}

In [0]:
# Check the actual column names in your Bronze DataFrame
print("Columns currently in Bronze:")
print(df_bronze.columns)

# Check the current data types
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, DateType

# 1. Define the columns we WANT to keep (this drops 'index' automatically)
# and ensures the types are correct.
df_silver = df_bronze.select(
    col("user_id").cast(StringType()),
    col("first_name").cast(StringType()),
    col("last_name").cast(StringType()),
    col("sex").cast(StringType()),
    col("email").cast(StringType()),
    col("phone").cast(StringType()),
    col("date_of_birth").cast(DateType()),
    col("job_title").cast(StringType())
)

# 2. Verify the result
print("Silver Schema (Index Dropped):")
df_silver.printSchema()

display(df_silver)

In [0]:
# Load the data from your Bronze table back into a DataFrame
df_bronze = spark.read.table("workspace.bronze.bronze_people") 

# Verify it loaded correctly
print(f"Bronze loaded. Row count: {df_bronze.count()}")

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, DateType

# 1. Load from Bronze
df_bronze = spark.read.table("workspace.bronze.bronze_people")

# 2. Transform (Select, Cast, and Drop Index)
df_silver = df_bronze.select(
    col("user_id").cast(StringType()),
    col("first_name").cast(StringType()),
    col("last_name").cast(StringType()),
    col("sex").cast(StringType()),
    col("email").cast(StringType()),
    col("phone").cast(StringType()),
    col("date_of_birth").cast(DateType()),
    col("job_title").cast(StringType())
)

# 3. Deduplicate
df_silver = df_silver.dropDuplicates(["user_id"])

# 4. Show result
display(df_silver)

In [0]:
# Define the target table path
target_table = "workspace.silver.silver_people"

# Save as a Delta table
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"Success! Your cleaned data is now live at: {target_table}")

In [0]:
%sql
OPTIMIZE workspace.silver.silver_people;
ANALYZE TABLE workspace.silver.silver_people COMPUTE STATISTICS;